### 1. Setup de Ambiente

In [0]:
catalog = "cinedata"
silver_schema = "silver"
gold_schema = "gold"

In [0]:
create_gold_schema = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}"
spark.sql(create_gold_schema)

### Fluxo
> Fiz primeiro as tabelas `dim`, por conta das SKs, depois fiz as tabelas `bridge` por precisarem das SKs das tabelas `dim`, e por fim a tabela `fact`, pois ela referencia `dim_movies`. Por fim, fiz a tabela GenAI.

### Dimensions

### 2. Tabela gold.dim_movies

In [0]:
from pyspark.sql import functions as F

# Dados da Silver
df_silver_info = spark.table(f"{catalog}.{silver_schema}.tb_info_filmes")

# Tabela Gold (utilizarei sha256 para a sk pois é determinístico e pode ser calculado sem o preço de um join na maioria das vezes)
df_gold_movies = (
    df_silver_info
    .withColumn("sk_movie_id", F.conv(F.substring(F.sha2(F.col("id_filme").cast("string"), 256), 1, 15), 16, 10).cast("bigint"))
    .withColumns({
        "id_filme": F.col("id_filme").cast("string"),
    })
).select("sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento", "duracao_minutos", "idioma_original", "status_filme", "sinopse")

# Salvo como tabela Delta
df_gold_movies.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.dim_movies")

### 3. Tabela gold.dim_genres

In [0]:
# Dados da Silver
df_silver_generos = spark.table(f"{catalog}.{silver_schema}.tb_generos")

# Tabela Gold como catálogo único (dropDuplicates(["genero"]))
df_gold_genres = (
    df_silver_generos
    .dropDuplicates(["genero"])
    .withColumn("sk_genre_id", F.conv(F.substring(F.sha2(F.col("genero"), 256), 1, 15), 16, 10).cast("bigint"))
    .withColumnRenamed("genero", "nome_genero")
).select("sk_genre_id", "nome_genero")

# Salvo como tabela Delta
df_gold_genres.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.dim_genres")

### 4. Tabela gold.dim_people

In [0]:
# Dados da Silver
df_silver_pessoas_empresas = spark.table(f"{catalog}.{silver_schema}.tb_pessoas_empresas")

# Tabela Gold filtrada para entidades de pessoas
df_gold_people = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
    .withColumn("sk_person_id", F.conv(F.substring(F.sha2(F.concat_ws("|", F.col("tipo_entidade"), F.col("nome")), 256), 1, 15), 16, 10).cast("bigint"))
    .withColumnsRenamed({
        "nome": "nome_pessoa",
        "tipo_entidade": "tipo_pessoa"
    })
).select("sk_person_id", "nome_pessoa", "tipo_pessoa")

# Salvo como tabela Delta
df_gold_people.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.dim_people")

### 5. Tabela gold.dim_companies

In [0]:
# Tabela Gold filtrada para Produtoras
df_gold_companies = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .dropDuplicates(["nome"])
    .withColumn("sk_company_id", F.conv(F.substring(F.sha2(F.col("nome"), 256), 1, 15), 16, 10).cast("bigint"))
    .withColumnRenamed("nome", "nome_produtora")
).select("sk_company_id", "nome_produtora")

# Salvo como tabela Delta
df_gold_companies.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.dim_companies")

### 6. Tabela gold.dim_reviews
> Creio que essa poderia ser uma tabela fato, já que é composta por métricas, e não por dados descritivos.

In [0]:
# Dados da Silver
df_silver_avaliacoes_usuarios = spark.table(f"{catalog}.{silver_schema}.tb_avaliacoes_usuarios")

# Tabela Gold com agregações e SKs
df_gold_reviews = (
    df_silver_avaliacoes_usuarios
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")
    )
    # Prefixei a sk_review_id para não confundir com a sk_movie_id, já que têm o mesmo grão
    .withColumn("sk_review_id", F.conv(F.substring(F.sha2(F.concat_ws("|", F.lit("review"), F.col("id_filme").cast("string")), 256), 1, 15), 16, 10).cast("bigint"))
    .withColumn("sk_movie_id", F.conv(F.substring(F.sha2(F.col("id_filme").cast("string"), 256), 1, 15), 16, 10).cast("bigint"))
).select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")

# Salvo como tabela Delta
df_gold_reviews.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.dim_reviews")

### Bridges

### 7. Tabela gold.bridge_movie_genre

In [0]:
df_bridge_movie_genre = (
    df_silver_generos
    .select(
        F.conv(F.substring(F.sha2(F.col("id_filme").cast("string"), 256), 1, 15), 16, 10).cast("bigint").alias("sk_movie_id"),
        F.conv(F.substring(F.sha2(F.col("genero"), 256), 1, 15), 16, 10).cast("bigint").alias("sk_genre_id")
    )
    .distinct()
)

df_bridge_movie_genre.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.bridge_movie_genre")

### 8. Tabela gold.bridge_movie_person

In [0]:
df_bridge_movie_person = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
    .select(
        F.conv(F.substring(F.sha2(F.col("id_filme").cast("string"), 256), 1, 15), 16, 10).cast("bigint").alias("sk_movie_id"),
        F.conv(F.substring(F.sha2(F.concat_ws("|", F.col("tipo_entidade"), F.col("nome")), 256), 1, 15), 16, 10).cast("bigint").alias("sk_person_id")
    )
    .distinct()
)

df_bridge_movie_person.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.bridge_movie_person")

### 9. Tabela gold.bridge_movie_company

In [0]:
df_bridge_movie_company = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        F.conv(F.substring(F.sha2(F.col("id_filme").cast("string"), 256), 1, 15), 16, 10).cast("bigint").alias("sk_movie_id"),
        F.conv(F.substring(F.sha2(F.col("nome"), 256), 1, 15), 16, 10).cast("bigint").alias("sk_company_id")
    )
    .distinct()
)

df_bridge_movie_company.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.bridge_movie_company")

### Fact

### 10. Tabela gold.fact_movies_performance

In [0]:
df_silver_financeiro = spark.table(f"{catalog}.{silver_schema}.tb_financeiro_filmes")
df_silver_metricas = spark.table(f"{catalog}.{silver_schema}.tb_metricas_engajamento")

df_silver_info = df_silver_info.filter(F.col("status_filme") == "Lançado")

df_fact_movies_performance = (
    df_silver_info
    .join(df_silver_financeiro, on="id_filme", how="left")
    .join(df_silver_metricas, on="id_filme", how="left")
).select(
    F.conv(F.substring(F.sha2(F.col("id_filme").cast("string"), 256), 1, 15), 16, 10).cast("bigint").alias("sk_movie_id"),
    F.col("orcamento_usd").try_cast("decimal(18,2)"),
    F.col("receita_usd").try_cast("decimal(18,2)"),
    F.col("lucro_usd").try_cast("decimal(18,2)"),
    F.col("orcamento_brl").try_cast("decimal(18,2)"),
    F.col("receita_brl").try_cast("decimal(18,2)"),
    F.col("lucro_brl").try_cast("decimal(18,2)"),
    F.col("popularidade").try_cast("double"),
    F.col("nota_media_tmdb").try_cast("double"),
    F.col("qtd_votos_tmdb").try_cast("integer"),
    F.col("nota_media_imdb").try_cast("double"),
    F.col("qtd_votos_imdb").try_cast("integer")
)

df_fact_movies_performance.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.fact_movies_performance")

### Tabela GenAI

### 11. Tabela gold_genai_movies_context

In [0]:
template = "O filme %s, lançado no ano de %s, faturou %s e teve um custo de %s. Estrelado por %s e dirigido por %s, o filme possui a seguinte sinopse: %s."

# Produtoras não entram no documento de contexto, portanto não utilizei a tabela dim_companies
df_elenco = (
    df_bridge_movie_person
    .join(df_gold_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(F.collect_set("nome_pessoa").alias("atores"))
)

df_diretores = (
    df_bridge_movie_person
    .join(df_gold_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(F.collect_set("nome_pessoa").alias("diretores"))
)

# Escolhi usar a receita e orçamento em reais, já que está em português
df_gold_genai = (
    df_gold_movies
    .join(df_fact_movies_performance, on="sk_movie_id", how="left")
    .join(df_elenco, on="sk_movie_id", how="left")
    .join(df_diretores, on="sk_movie_id", how="left")
    .withColumn("movie_id", F.col("id_filme"))
    .withColumn("title", F.col("titulo"))
    .withColumn("llm_context_document", F.format_string(
        template,
        F.coalesce(F.col("titulo"), F.lit("de título não informado")),
        F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("valor desconhecido")),
        F.coalesce(F.format_number(F.col("receita_brl"), 2), F.lit("uma receita em reais não divulgada")),
        F.coalesce(F.format_number(F.col("orcamento_brl"), 2), F.lit("orçamento em reais não divulgado")),
        F.when(F.size(F.coalesce(F.col("atores"), F.array())) > 0, 
               F.concat_ws(", ", F.col("atores")))
        .otherwise(F.lit("um elenco não informado")),
        F.when(F.size(F.coalesce(F.col("diretores"), F.array())) > 0,
               F.concat_ws(", ", F.col("diretores")))
        .otherwise(F.lit("uma direção não informada")),
        F.coalesce(F.col("sinopse"), F.lit("Sinopse não disponível."))
    ))
).select("movie_id", "title", "llm_context_document")

df_gold_genai.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.genai_movies_context")

### Perguntas de Negócio

In [0]:
%sql
SELECT ROUND(SUM(receita_brl), 2) AS receita_total_brl 
FROM cinedata.gold.fact_movies_performance

In [0]:
%sql
SELECT m.titulo, mp.popularidade 
FROM cinedata.gold.fact_movies_performance mp
JOIN cinedata.gold.dim_movies m
ON mp.sk_movie_id = m.sk_movie_id
ORDER BY mp.popularidade DESC
LIMIT 5

In [0]:
%sql
SELECT nome_genero AS genero, COUNT(bg.sk_movie_id) AS qtd_filmes
FROM cinedata.gold.dim_genres AS g
JOIN cinedata.gold.bridge_movie_genre AS bg
ON g.sk_genre_id = bg.sk_genre_id
GROUP BY g.nome_genero

In [0]:
%sql
WITH ranking_receita AS (
    SELECT 
        m.titulo, 
        mp.receita_usd, 
        mp.receita_brl, 
        RANK() OVER (ORDER BY mp.receita_usd DESC) AS posicao_ranking
    FROM cinedata.gold.fact_movies_performance AS mp
    JOIN cinedata.gold.dim_movies AS m
    ON mp.sk_movie_id = m.sk_movie_id
    WHERE receita_usd IS NOT NULL OR receita_brl IS NOT NULL
)
SELECT titulo, receita_usd, receita_brl, posicao_ranking
FROM ranking_receita
WHERE posicao_ranking <= 10
ORDER BY posicao_ranking

In [0]:
%sql
WITH filmes_lancados AS (
    SELECT sk_movie_id, ano_lancamento
    FROM cinedata.gold.dim_movies
    WHERE status_filme = 'Lançado'
      AND ano_lancamento IS NOT NULL
      AND ano_lancamento <= YEAR(CURRENT_DATE())
),
data_maxima AS (
    SELECT MAX(ano_lancamento) AS ano_maximo
    FROM filmes_lancados
)
SELECT
    p.nome_pessoa AS ator,
    COUNT(bmp.sk_movie_id) AS qtd_participacoes
FROM cinedata.gold.bridge_movie_person AS bmp
JOIN cinedata.gold.dim_people AS p
    ON bmp.sk_person_id = p.sk_person_id
JOIN filmes_lancados AS m
    ON bmp.sk_movie_id = m.sk_movie_id
JOIN data_maxima AS dm
    ON m.ano_lancamento >= dm.ano_maximo - 1
WHERE p.tipo_pessoa = 'Ator'
GROUP BY p.nome_pessoa
ORDER BY qtd_participacoes DESC
LIMIT 1

In [0]:
%sql
WITH filmes_lancados AS (
    SELECT sk_movie_id, ano_lancamento
    FROM cinedata.gold.dim_movies
    WHERE status_filme = 'Lançado'
      AND ano_lancamento IS NOT NULL
      AND ano_lancamento <= YEAR(CURRENT_DATE())
),
data_maxima AS (
    SELECT MAX(ano_lancamento) AS ano_maximo
    FROM filmes_lancados
),
filmes_periodo AS (
    SELECT f.sk_movie_id
    FROM filmes_lancados f
    JOIN data_maxima dm
      ON f.ano_lancamento >= dm.ano_maximo - 4
)
SELECT
    c.nome_produtora AS produtora,
    SUM(mp.lucro_usd) AS lucro_total_usd,
    SUM(mp.lucro_brl) AS lucro_total_brl
FROM cinedata.gold.bridge_movie_company AS bmc
JOIN cinedata.gold.dim_companies AS c
    ON bmc.sk_company_id = c.sk_company_id
JOIN filmes_periodo AS fp
    ON bmc.sk_movie_id = fp.sk_movie_id
JOIN cinedata.gold.fact_movies_performance AS mp
    ON bmc.sk_movie_id = mp.sk_movie_id
GROUP BY c.nome_produtora
ORDER BY lucro_total_usd DESC
LIMIT 1